In [106]:
import pandas as pd
import numpy as np
import random
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [107]:
df = pd.read_csv("india_women_safety_risk_dataset.csv")

df.head()


,latitude,longitude,hour,crime_density,poi_count,is_night,is_isolated,risk_label
0,23.256181,77.435799,17,3.8,5,0,0,0
1,23.342607,77.470815,9,5.1,6,0,0,1
2,23.309799,77.464024,10,1.2,5,0,0,0
3,23.289799,77.373085,16,1.2,7,0,0,0
4,23.223403,77.372387,23,2.5,6,1,0,0


In [108]:

print("Columns & types:\n", df.info())
print("\nFirst 5 rows:\n", df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   latitude       4000 non-null   float64
 1   longitude      4000 non-null   float64
 2   hour           4000 non-null   int64  
 3   crime_density  4000 non-null   float64
 4   poi_count      4000 non-null   int64  
 5   is_night       4000 non-null   int64  
 6   is_isolated    4000 non-null   int64  
 7   risk_label     4000 non-null   int64  
dtypes: float64(3), int64(5)
memory usage: 250.1 KB
Columns & types:
 None

First 5 rows:
     latitude  longitude  hour  crime_density  poi_count  is_night  \
0  23.256181  77.435799    17            3.8          5         0   
1  23.342607  77.470815     9            5.1          6         0   
2  23.309799  77.464024    10            1.2          5         0   
3  23.289799  77.373085    16            1.2          7         0   
4  23.223403  77.3

In [109]:
# =========================
# Step 3: Balance dataset (optional but recommended)
# =========================
# Upsample HIGH risk rows if needed
low = df[df.risk_label==0].sample(2000, replace=True, random_state=42)
medium = df[df.risk_label==1].sample(2000, replace=True, random_state=42)
high = df[df.risk_label==2].sample(2000, replace=True, random_state=42)

df_balanced = pd.concat([low, medium, high]).reset_index(drop=True)
print("\nBalanced Risk Label Distribution:")
print(df_balanced["risk_label"].value_counts())



Balanced Risk Label Distribution:
risk_label
0    2000
1    2000
2    2000
Name: count, dtype: int64


In [110]:
# =========================
# Step 4: Train-test split
# =========================
X = df_balanced.drop(["risk_label", "is_night", "is_isolated"], axis=1)
y = df_balanced["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [111]:
# =========================
# Step 5: Train XGBoost model
# =========================
model = XGBClassifier(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=120, n_jobs=None,
              num_parallel_tree=None, ...)

In [112]:
# =========================
# Step 6: Evaluate model
# =========================
y_pred = model.predict(X_test)
print("\nAccuracy on test set:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))



Accuracy on test set: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00       400
           2       1.00      1.00      1.00       400

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200



In [113]:
# =========================
# Step 7: Define hybrid risk prediction function
# =========================
def predict_risk_final(model, input_df):
    """
    Hybrid Rule + XGBoost risk scoring:
    - Rule-based component ensures realistic scoring
    - XGBoost refines probability prediction
    """
    row = input_df.iloc[0]
    # Rule-based score
    score = 0
    if row['hour'] >= 22 or row['hour'] <= 4:
      score += 35   # late night (dangerous)
    elif row['hour'] >= 18:
      score += 20   # evening

    threshold = df_balanced.poi_count.quantile(0.2)

    if row['poi_count'] < threshold:
        score += 30
    elif row['poi_count'] < threshold * 2:
        score += 15

    # Crime impact
    if row['crime_density'] < 2:
        score += row['crime_density'] * 8
    elif row['crime_density'] < 4:
        score += row['crime_density'] * 12
    else:
        score += row['crime_density'] * 18
        score = min(score, 100)  # cap at 100

    # XGBoost probability adjustment
    probs = model.predict_proba(input_df)
    weights = [0, 70, 100]
    xgb_score = sum(p * w for p, w in zip(probs[0], weights))

    # Combine rule + model (weighted average)
    final_score = int(0.6*score + 0.4*xgb_score)

    # Map to Risk Level
    if final_score < 30:
        level = "LOW"
    elif final_score < 65:
        level = "MEDIUM"
    else:
        level = "HIGH"
    return final_score, level

In [114]:
# =========================
# Step 8: Generate 10 realistic random samples (FIXED)
# =========================

threshold = df_balanced.poi_count.quantile(0.2)

random_samples = []

for _ in range(10):
    # Hour
    hour = random.randint(0, 23)

    # More realistic POI distribution (not uniform)
    poi_count = int(np.random.normal(loc=10, scale=5))
    poi_count = max(0, poi_count)

    # Crime density with slight correlation
    crime_density = round(random.uniform(
        df_balanced.crime_density.min(),
        df_balanced.crime_density.max()
    ), 1)

    # Add correlation: low POI → slightly higher crime
    if poi_count < threshold:
        crime_density += random.uniform(1, 2)

    # Location
    latitude = round(random.uniform(
        df_balanced.latitude.min(),
        df_balanced.latitude.max()
    ), 6)

    longitude = round(random.uniform(
        df_balanced.longitude.min(),
        df_balanced.longitude.max()
    ), 6)

    # Append ONLY required features
    random_samples.append({
        "latitude": latitude,
        "longitude": longitude,
        "hour": hour,
        "crime_density": crime_density,
        "poi_count": poi_count
    })

random_samples = pd.DataFrame(random_samples)

In [115]:
# =========================
# Step 9: Predict Risk for Random Samples
# =========================
print("\nRandom Sample Predictions:\n")
for i, row in random_samples.iterrows():
    sample_df = pd.DataFrame([row])
    score, level = predict_risk_final(model, sample_df)
    print(f"Sample {i+1}: Risk Score = {score}, Risk Level = {level}")


Random Sample Predictions:

Sample 1: Risk Score = 33, Risk Level = MEDIUM
Sample 2: Risk Score = 26, Risk Level = LOW
Sample 3: Risk Score = 27, Risk Level = LOW
Sample 4: Risk Score = 48, Risk Level = MEDIUM
Sample 5: Risk Score = 49, Risk Level = MEDIUM
Sample 6: Risk Score = 88, Risk Level = HIGH
Sample 7: Risk Score = 99, Risk Level = HIGH
Sample 8: Risk Score = 27, Risk Level = LOW
Sample 9: Risk Score = 87, Risk Level = HIGH
Sample 10: Risk Score = 46, Risk Level = MEDIUM


In [116]:
# =========================
# LOW-RISK TEST CASE
# =========================

import pandas as pd

# Create a manual LOW-risk sample
low_risk_sample = pd.DataFrame([{
    "latitude": 23.300000,        # example location
    "longitude": 77.400000,
    "hour": 14,                   # daytime
    "crime_density": 1.0,         # very low
    "poi_count": 15,              # many nearby POIs
}])

# Predict Risk Score & Risk Level
score, level = predict_risk_final(model, low_risk_sample)

# Print the results
print("Low-Risk Test Case:")
print("Risk Score =", score)
print("Risk Level =", level)


Low-Risk Test Case:
Risk Score = 4
Risk Level = LOW


In [120]:
# =========================
# TEST CASES: LOW, MEDIUM, HIGH
# =========================

import pandas as pd
# -------------------------
# 1️⃣ LOW RISK sample (clearly safe)
# -------------------------
low_risk_sample = pd.DataFrame([{
    "latitude": 23.300000,
    "longitude": 77.400000,
    "hour": 13,          # daytime
    "crime_density": 0.5, # very low
    "poi_count": 18,     # crowded
}])

# -------------------------
# 2️⃣ MEDIUM RISK sample (clearly risky)
# -------------------------
medium_risk_sample = pd.DataFrame([{
    "latitude": 23.310000,
    "longitude": 77.410000,
    "hour": 21,          # night
    "crime_density": 4.0, # stronger
    "poi_count": 4,      # semi-isolated
}])

# -------------------------
# 3️⃣ HIGH RISK sample (extreme case)
# -------------------------
high_risk_sample = pd.DataFrame([{
    "latitude": 23.320000,
    "longitude": 77.420000,
    "hour": 23,          # late night
    "crime_density": 6.5, # very high
    "poi_count": 0,      # extreme isolation
}])
# -------------------------
# Run predictions
# -------------------------
predict_case("LOW-RISK", low_risk_sample)
predict_case("MEDIUM-RISK", medium_risk_sample)
predict_case("HIGH-RISK", high_risk_sample)


LOW-RISK Test Case:
Risk Score = 2
Risk Level = LOW
------------------------------
MEDIUM-RISK Test Case:
Risk Score = 60
Risk Level = MEDIUM
------------------------------
HIGH-RISK Test Case:
Risk Score = 88
Risk Level = HIGH
------------------------------


In [118]:
import joblib


In [119]:
# Save the trained XGBoost model
joblib.dump(model, "women_safety_xgb_model.pkl")
print("Model saved as women_safety_xgb_model.pkl")


Model saved as women_safety_xgb_model.pkl


In [121]:
import pandas as pd

real_life_cases = [
    # 1. Safe daytime market
    {
        "scenario": "Busy market in daytime",
        "latitude": 23.30,
        "longitude": 77.40,
        "hour": 12,
        "crime_density": 0.5,
        "poi_count": 20
    },

    # 2. College campus afternoon
    {
        "scenario": "College campus afternoon",
        "latitude": 23.31,
        "longitude": 77.41,
        "hour": 15,
        "crime_density": 1.0,
        "poi_count": 18
    },

    # 3. Office area evening rush
    {
        "scenario": "Office area evening rush",
        "latitude": 23.32,
        "longitude": 77.42,
        "hour": 18,
        "crime_density": 2.5,
        "poi_count": 15
    },

    # 4. Walking home at 9 PM
    {
        "scenario": "Walking home at night",
        "latitude": 23.33,
        "longitude": 77.43,
        "hour": 21,
        "crime_density": 4.0,
        "poi_count": 6
    },

    # 5. Waiting alone at bus stop
    {
        "scenario": "Waiting alone at bus stop",
        "latitude": 23.34,
        "longitude": 77.44,
        "hour": 22,
        "crime_density": 4.5,
        "poi_count": 3
    },

    # 6. Empty street late night
    {
        "scenario": "Empty street late night",
        "latitude": 23.35,
        "longitude": 77.45,
        "hour": 23,
        "crime_density": 5.5,
        "poi_count": 1
    },

    # 7. Cab ride through isolated road
    {
        "scenario": "Cab on isolated road",
        "latitude": 23.36,
        "longitude": 77.46,
        "hour": 1,
        "crime_density": 5.0,
        "poi_count": 0
    },

    # 8. Residential area early morning
    {
        "scenario": "Residential area early morning",
        "latitude": 23.37,
        "longitude": 77.47,
        "hour": 6,
        "crime_density": 2.0,
        "poi_count": 10
    },

    # 9. Festival crowd at night
    {
        "scenario": "Festival crowd at night",
        "latitude": 23.38,
        "longitude": 77.48,
        "hour": 20,
        "crime_density": 3.5,
        "poi_count": 25
    },

    # 10. Highway breakdown at night
    {
        "scenario": "Highway breakdown at night",
        "latitude": 23.39,
        "longitude": 77.49,
        "hour": 23,
        "crime_density": 6.5,
        "poi_count": 0
    }
]

df_cases = pd.DataFrame(real_life_cases)

In [122]:
for i, row in df_cases.iterrows():
    sample_df = pd.DataFrame([row.drop("scenario")])

    score, level = predict_risk_final(model, sample_df)

    print(f"Scenario: {row['scenario']}")
    print(f"Risk Score = {score}, Risk Level = {level}")
    print("-"*40)

Scenario: Busy market in daytime
Risk Score = 2, Risk Level = LOW
----------------------------------------
Scenario: College campus afternoon
Risk Score = 4, Risk Level = LOW
----------------------------------------
Scenario: Office area evening rush
Risk Score = 30, Risk Level = MEDIUM
----------------------------------------
Scenario: Walking home at night
Risk Score = 55, Risk Level = MEDIUM
----------------------------------------
Scenario: Waiting alone at bus stop
Risk Score = 60, Risk Level = MEDIUM
----------------------------------------
Scenario: Empty street late night
Risk Score = 88, Risk Level = HIGH
----------------------------------------
Scenario: Cab on isolated road
Risk Score = 60, Risk Level = MEDIUM
----------------------------------------
Scenario: Residential area early morning
Risk Score = 14, Risk Level = LOW
----------------------------------------
Scenario: Festival crowd at night
Risk Score = 37, Risk Level = MEDIUM
----------------------------------------
